In [3]:
import pandas as pd
from bs4 import BeautifulSoup
import re

In [4]:
df = pd.read_csv(r'D:\nlp-basics\Text Representaion\IMDB Dataset.csv')

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


## data cleaning

* remove websites first
* it contains html tags
* there are words like this: Halliwell\'s :we need to convert it into like: Halliwell's :this
* lowercaing
* check for emojis then remove
* removing urls check first
* stopwords
* tokenization
* stemming

In [7]:
#remove website links (not perfext but better in context of data)
import re
def remove_website_links(text):
    pattern = 'https?:(\S| )\S+'
    data = re.sub(pattern,'',text)
    return data

df['review'] = df['review'].apply(remove_website_links)

In [8]:
df[df['review'].str.contains('http')]


,review,sentiment


In [9]:
def remove_html_tags(text):
    if isinstance(text,str):
        soup = BeautifulSoup(text,'html.parser')
        clean_text = soup.get_text(separator=' ',strip=True)
    else:
        pass
    return clean_text

df['review'] = df['review'].apply(remove_html_tags)

C:\Users\bhosa\AppData\Local\Temp\ipykernel_3264\2213511721.py:3: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text,'html.parser')


In [10]:
def remove_puncuations(text):
    puncs = '"$%&()*+,-.!/:;<=>?@[\\]^_`{|}~'
    return text.translate(str.maketrans("","",puncs))

df['review'] = df['review'].apply(remove_puncuations)

In [11]:
import emoji
def convert_emoji_text(text):
    clean_text = emoji.demojize(text)
    clean_text = clean_text.replace(":","")
    clean_text = clean_text.replace("_","")
    return clean_text

df['review'] = df['review'].apply(convert_emoji_text)

#### lingua

In [12]:
'''from lingua import Language,LanguageDetectorBuilder
languages = [Language.ENGLISH, Language.FRENCH, Language.GERMAN, Language.SPANISH,Language.CHINESE,Language.DANISH]
detector = LanguageDetectorBuilder.from_languages(*languages).build()

def remove_other_language_reviews(text):
    try: 
        if detector.detect_language_of(text) == Language.ENGLISH:
            return text
        else:
            return np.nan
    except:
        return 'cool'
    
df['lingua']  =df['review'].apply(remove_other_language_reviews) '''
#not neccesary

"from lingua import Language,LanguageDetectorBuilder\nlanguages = [Language.ENGLISH, Language.FRENCH, Language.GERMAN, Language.SPANISH,Language.CHINESE,Language.DANISH]\ndetector = LanguageDetectorBuilder.from_languages(*languages).build()\n\ndef remove_other_language_reviews(text):\n    try: \n        if detector.detect_language_of(text) == Language.ENGLISH:\n            return text\n        else:\n            return np.nan\n    except:\n        return 'cool'\n\ndf['lingua']  =df['review'].apply(remove_other_language_reviews) "

## corpus

In [13]:
total_words = []
for sent in df['review'].tolist():
    words =  sent.split(' ')
    total_words.extend(words)

In [14]:
unique_words = set(total_words)
print(f'total words in df[review] {len(unique_words)}')

total words in df[review] 214983


## BOW

In [15]:
import spacy
from nltk.stem.porter import PorterStemmer

nlp  = spacy.load('en_core_web_sm',disable=['parser','tagger','ner','tok2vec'])
ps = PorterStemmer()

def spacy_tokenizer(data_list):
    updated_text = []
    for doc in nlp.pipe(data_list,batch_size=500):       
        tokens = [ps.stem(token.text)  if token.is_alpha else token.text for token in doc]
        updated_text.append(" ".join(tokens))
    return updated_text

In [16]:
updated_text = spacy_tokenizer(df['review'].tolist())

c:\python\Lib\site-packages\spacy\pipeline\lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [17]:
df['stem_tokenized_text'] = updated_text

In [18]:
df.head()

,review,sentiment,stem_tokenized_text
0,One of the other reviewers has mentioned that ...,positive,one of the other review ha mention that after ...
1,A wonderful little production The filming tech...,positive,a wonder littl product the film techniqu is ve...
2,I thought this was a wonderful way to spend ti...,positive,i thought thi wa a wonder way to spend time on...
3,Basically there's a family where a little boy ...,negative,basic there 's a famili where a littl boy jake...
4,Petter Mattei's Love in the Time of Money is a...,positive,petter mattei 's love in the time of money is ...


In [19]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
cv = CountVectorizer(stop_words ="english")

In [20]:
bow  = cv.fit_transform(df['stem_tokenized_text'])

In [21]:
import numpy as np
np.set_printoptions(threshold=np.inf)
bow[0].toarray().shape

(1, 127215)

## N- grams

In [22]:
from sklearn.feature_extraction.text import CountVectorizer
ngram = CountVectorizer()

In [23]:
grams = ngram.fit_transform(df['stem_tokenized_text'])

In [24]:
grams[0].shape

(1, 127466)

## TF-IDF

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

In [26]:
tf = tfidf.fit_transform(df['review'])

In [27]:
tf.shape

(49582, 159052)

In [28]:
tf_df = pd.DataFrame({
    'words':tfidf.get_feature_names_out(),
    'tf-idf':tfidf.idf_
})

In [29]:
tf_df.sample(12)

,words,tf-idf
98552,oafishness,11.118256
2884,678,11.118256
101915,overdoses,9.731962
120574,sainthood,10.425109
59986,grandma,7.429377
92815,moviethe,7.841111
15146,beeror,11.118256
155357,wireframe,11.118256
10004,aquacom,11.118256
48024,exflame,10.201965
